In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
import os
from zoneinfo import ZoneInfo
from typing import Iterable, Union, Tuple, List
from pathlib import Path

from frequenz.data.microgrid import component_data
from frequenz.gridpool.config import MicrogridConfig, load_configs
import frequenz.lib.notebooks.reporting.plotter as pl
from frequenz.client.common.metric import Metric        
from frequenz.client.reporting import ReportingApiClient
from frequenz.client.assets import AssetsApiClient
from frequenz.lib.notebooks.reporting.utils.column_mapper import ColumnMapper
from frequenz.lib.notebooks.reporting.utils.helpers import normalize_date_for_reporting, set_date_to_midnight, add_energy_flows, get_meter_display_names
from frequenz.lib.notebooks.reporting.data_processing import create_energy_report_df
from frequenz.lib.notebooks.reporting.utils.reporting_nb_functions import build_overview_df, compute_energy_summary, aggregate_metrics, build_component_analysis, assemble_component_analysis
from frequenz.lib.notebooks.reporting.plotter import plot_time_series, plot_time_series_battery_soc_and_usecase

from frequenz.lib.notebooks.reporting.asset_optimization.data import merge_day_ahead_prices
from IPython.display import display, Markdown

import re
import numpy as np

import logging
logging.basicConfig()
logging.getLogger("frequenz.lib.notebooks").setLevel(logging.WARNING)

pv_analyse_df, bat_analyse_df, chp_analyse_df, wind_analyse_df, ev_analyse_df = (pd.DataFrame() for _ in range(5))
ev_prod_filter_sum, wind_prod_filter_sum, chp_prod_filter_sum, bat_prod_filter_sum, pv_prod_filter_sum, pv_production_sum, net_site_consumption_sum, grid_consumption_sum, pv_feed_in_sum, peak, pv_self_consumption_sum, pv_self_consumption_share, pv_total_consumption_share = (0 for _ in range(13))

battery_soc_df = pd.DataFrame()
import warnings
warnings.filterwarnings("ignore")

def component_id(col: str, component_type: str) -> str | None:
    """Extract the numeric ID from a component column name.

    Examples:
        component_id("PV #3 inv active_power", "PV") -> "3"
        component_id("Battery #12 soc", "Battery") -> "12"
    """
    match = re.match(rf"^{re.escape(component_type)} #(\d+)\b", col)
    return match.group(1) if match else None

In [ ]:
color_dict = {
    "PV": "rgba(255,243,138,1)",
#     "PV-Erzeugung": "rgba(255,243,138,1)",
#     "PV-Erzeugung [kWh]": "rgba(255,243,138,1)",
#     "PV-Erzeugung [kWh] Sum": "rgba(255,243,138,1)",
#     "Wind": "rgba(100,149,237,1)",
#     "Wind-Erzeugung": "rgba(100,149,237,1)",
#     "Wind-Erzeugung [kWh] Sum": "rgba(100,149,237,1)",
#     "BHKW": "rgba(255,140,0,1)",
#     "BHKW-Erzeugung": "rgba(255,140,0,1)",
#     "BHKW-Erzeugung [kWh] Sum": "rgba(255,140,0,1)",
#     "Netto Gesamtverbrauch": "rgba(70,70,70,1)",
#     "MID Gesamtverbrauch": "rgba(70,70,70,1)",
#     "Batterie": "rgba(0,204,150,1)",
#     "Netzbezug": "rgba(0,0,0,1)",
    "Grid Consumption": "rgba(0,0,0,1)",
}

europe_timezones = [
    "UTC",
    "Europe/Berlin",
    "Europe/Vienna",
    "Europe/Zurich",
    "Europe/London",
]

# Reporting Notebook

⚠️ **Wichtig:**
Um die Anweisungen anzuzeigen (falls nicht bereits angezeigt), müssen Sie **die nächste Zelle ausführen**.

▶️ **Wie führe ich die nächste Zelle aus?**
--> Drücken Sie die **Run** ▶️-Taste oben.


In [ ]:
%%html
<style>
  details {
    background: #f8f9fa;
    border-radius: 6px;
    padding: 12px;
    margin-bottom: 12px;
    border: 1px solid #ddd;
  }

  summary {
    font-size: 18px;
    font-weight: bold;
    cursor: pointer;
    padding: 6px;
    border-radius: 4px;
  }

  summary:hover {
    color: #007bff;
  }

  p {
    font-size: 15px;
    line-height: 1.6;
  }
  
  ul {
    margin-left: 20px;
    padding-left: 15px;
    font-size: 15px;
  }
  
  li {
    margin-bottom: 8px;
  }
  
  /* Style checked items */
  .checked {
    text-decoration: line-through;
    color: gray;
  }
</style>

<details>
  <summary>🇩🇪 <strong>Deutsche Anleitung (Klicken zum Anzeigen)</strong></summary>

  <p><strong>🛠️ Anleitung</strong></p>
  <p>Das Notebook benötigt einige Einstellungen, bevor es zum ersten Mal ausgeführt werden kann.</p>

  <ul>
    <li><input type="checkbox" onclick="toggleStrike(this)"> ✅ <strong>Neueste Paketversion sicherstellen:</strong> Es wird empfohlen, die neuesten Version der benötigten Pakete zu installieren. Alle Versionen finden Sie hier: <a href="https://pypi.org/project/frequenz-lib-notebooks/#history">frequenz-lib-notebooks</a>, <a href="https://pypi.org/project/frequenz-client-reporting/#history">frequenz-client-reporting</a>. Ersetzen Sie bei der Installation die vordefinierte Version durch die gewünschte.</li>
    <li><input type="checkbox" onclick="toggleStrike(this)"> 🛠️ <strong>Führen Sie die erste Code-Zelle aus:</strong> Klicken Sie auf das <strong><span style="color: blue;">blaue</span> Dreieck</strong>, um Pakete zu installieren. Danach verschieben Sie sie zu <strong>"requirements.txt"</strong> (weitere Informationen finden Sie im Dokument <em>"Anleitung: Deepnote Projekt erstellen und Notebook importieren"</em>, Abschnitt <em>"5. Requirements installieren"</em>).</li>
    <li><input type="checkbox" onclick="toggleStrike(this)"> 📦 <strong>Importieren Sie die Bibliotheken:</strong> Führen Sie die nächsten zwei Code-Zellen aus, um die notwendigen Bibliotheken zu importieren und die Datei <code>microgrids.toml</code> zu laden. ⏳ <em>Warten Sie, bis die Ausführung abgeschlossen ist.</em></li>
    <li><input type="checkbox" onclick="toggleStrike(this)"> 🎯 <strong>Konfigurieren Sie die Eingaben:</strong> Füllen Sie die erforderlichen Felder unter <strong>"Input"</strong> aus. </li>
    <li><input type="checkbox" onclick="toggleStrike(this)"> 🚀 <strong>Notebook ausführen:</strong> Klicken Sie auf den <strong><span style="color: blue;">"Start"</span>-Button</strong>, sobald alle Eingaben ausgefüllt sind.</li>
  </ul>

</details>

<script>
  function toggleStrike(checkbox) {
    if (checkbox.checked) {
      checkbox.parentElement.classList.add("checked");
    } else {
      checkbox.parentElement.classList.remove("checked");
    }
  }
</script>

# Einleitung

In diesem Notebook können Sie die Daten aus der Reporting API analysieren und visualisieren\. Sie haben die Möglichkeit, den gewünschten Zeitraum und die Auflösung der zu analysierenden Daten festzulegen\.
Die Analyse umfasst wichtige Kennzahlen zur Stromgewinnung aus Photovoltaikanlagen \(PV\), zur Nutzung von Batteriespeichern sowie zum Netzanschluss\. Die Ergebnisse werden in anschaulichen Charts dargestellt, um Ihnen einen klaren Überblick über die Energieflüsse zu ermöglichen\.
Wählen Sie einfach Ihre gewünschten Parameter aus, um tiefere Einblicke in Ihre Energieerzeugung und \-nutzung zu erhalten\.

# Eingaben

In [ ]:
assets_client = AssetsApiClient(
    server_url=os.environ["ASSETS_API_URL"],
    auth_key=os.environ["FREQUENZ_API_KEY"],
    sign_secret=os.environ["FREQUENZ_API_SECRET"],
)

assets_logger = logging.getLogger("frequenz.client.assets")
config_logger = logging.getLogger("frequenz.gridpool.config.load")
old_assets_level = assets_logger.level
old_config_level = config_logger.level
assets_logger.setLevel(logging.ERROR)
config_logger.setLevel(logging.ERROR)

directory = "toml_directory/"
configs = await load_configs(
    list(Path(directory).glob("*.toml")),
    assets_client=assets_client,

    )

available_microgrids = list(configs.microgrids.keys())

# assets_logger.setLevel(old_assets_level)
# config_logger.setLevel(old_config_level)

Um das Notebook nutzen zu können müssen zunächst die API Anmeldeinformationen hinterlegt werden \(Anleitung\)\. Die API Anmeldeinformationen finden Sie im Kuiper\-Portal\. Dort finden Sie ebenfalls ihre Microgrid ID, die Meter PQ ID, die Meter Batterie IDs sowie die Meter PV IDs\. Bitte geben Sie Ihre Komponenten IDs unten ein, wählen sie ein Start\- und Enddatum, wählen sie eine Auflösung und klicken Sie auf Start, um das Notebook auszuführen\.
Eine Anleitung zum Vorgehen finden Sie hier\.

In [ ]:
microgrid_id = '242'

Zeitrahmen

In [ ]:

from dateutil.parser import parse as _deepnote_parse
start_date = _deepnote_parse('2026-08-01T00:00:00.000Z').date()


In [ ]:

from dateutil.parser import parse as _deepnote_parse
end_date = _deepnote_parse('2026-08-31T00:00:00.000Z').date()


In [ ]:
timezone = 'Europe/Berlin'

In [ ]:
resolution = '15'

# 

# 

# 

In [ ]:
mapper = ColumnMapper.from_default()
start_date = set_date_to_midnight(start_date, timezone)
end_date = normalize_date_for_reporting(end_date, timezone) - timedelta(hours=2)
end_date = end_date.replace(microsecond=0)
today = date.today()
if end_date.date() != today:
    end_date += timedelta(days=1)

processed_resolution = timedelta(seconds=int(resolution) * 60)
timestep_hours = int(resolution) / 60
microgrid_id = int(microgrid_id)
print(f"Microgrid ID: {microgrid_id}")
print(f"Start Date: {start_date}")
print(f"End Date: {end_date}")
print(f"Resolution: {processed_resolution}")

In [ ]:
mdata = component_data.MicrogridData(
    server_url=os.environ["REPORTING_API_URL"],
    auth_key=os.environ["FREQUENZ_API_KEY"],
    sign_secret=os.environ["FREQUENZ_API_SECRET"],
    # microgrid_configs=mg_config,
    microgrid_configs=configs,
) 

# mids = [i.replace("iot", "") for i in mdata.microgrid_ids] 

mcfg = mdata.microgrid_configs[microgrid_id]
ctypes = mcfg.component_types()
component_types = ctypes

df = await mdata.ac_active_power(
    microgrid_id=microgrid_id, 
    component_types=component_types,
    start=start_date,
    end=end_date, # To include the data from the end date
    resampling_period=processed_resolution,
    keep_components=True,
    splits=True,
)

if 'battery' in component_types and "BATTERY_SOC_PCT" in mcfg.ctype["battery"].formula:
    battery_soc_df = await mdata.soc(
        microgrid_id=microgrid_id,
        start=start_date,
        end=end_date, # To include the data from the end date
        resampling_period=processed_resolution,
    )

if df is not None and not df.empty:
    # Remove the component_types where the meters are not working
    present = [c for c in component_types if c in df.columns]

    # Optionally, see which were missing
    missing = [c for c in component_types if c not in df.columns]
    if missing:
        print(f"Skipping missing components (no column in df): {missing}")

    # Now drop components whose meters sum to zero
    component_types = [
        c for c in present
        if pd.to_numeric(df[c], errors="coerce").fillna(0).sum() != 0
    ]

    print(f"Received {df.shape[0]} rows and {df.shape[1]} columns")

    # df.to_csv('raw_df.csv')

df

In [ ]:
if "ev_asset_production" not in df.columns:
    df["ev_asset_production"] = 0.0

component_display_names = await get_meter_display_names(int(mcfg.microgrid_id))
energy_report_df = create_energy_report_df(df, component_types,
                                           mcfg, mapper=mapper, tz_name=timezone,
                                           assume_tz="UTC", 
                                           component_display_names=component_display_names,
                                           battery_soc_df=battery_soc_df
)

try:
    energy_report_df = merge_day_ahead_prices(
        energy_report_df.set_index("timestamp"),
        dayahead_country_code="DE_LU",
    ).reset_index()
except (ModuleNotFoundError, ValueError) as err:
    print(f"Skipping day-ahead price integration: {err}")

energy_metrics_dict = aggregate_metrics(energy_report_df=energy_report_df, 
resolution=processed_resolution, 
price_column="day_ahead_price")

pv_production_sum = energy_metrics_dict["pv_production_sum"]
chp_production_sum = energy_metrics_dict["chp_production_sum"]
wind_production_sum = energy_metrics_dict["wind_production_sum"]
prod_self_consumption_sum = energy_metrics_dict["prod_self_consumption_sum"]
prod_bat_sum = energy_metrics_dict["prod_bat_sum"]
grid_feed_in_sum = energy_metrics_dict["grid_feed_in_sum"]
grid_import_sum = energy_metrics_dict["grid_consumption_sum"]
mid_consumption_sum = energy_metrics_dict["mid_consumption_sum"]
total_production_sum = energy_metrics_dict["total_production_sum"]
prod_self_consumption_share = energy_metrics_dict["prod_self_consumption_share"]
prod_self_production_share = energy_metrics_dict["prod_self_production_share"]
peak = energy_metrics_dict["peak"]
peak_date = energy_metrics_dict["peak_date"]
grid_import_cost_sum = energy_metrics_dict["grid_import_cost_sum"]
grid_feed_in_revenue_sum = energy_metrics_dict["grid_feed_in_revenue_sum"]
production_to_battery_sum = energy_metrics_dict["production_to_battery_sum"]
grid_to_battery_sum = energy_metrics_dict["grid_to_battery_sum"]
battery_to_grid_sum = energy_metrics_dict["battery_to_grid_sum"]
battery_to_consumption_sum = energy_metrics_dict["battery_to_consumption_sum"]

energy_report_df

# Übersicht

In [ ]:
print(f'{datetime.strftime(start_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")} - {datetime.strftime(end_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")}')

### Netzkennzahlen

In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Bezug in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{grid_import_sum}"
    })

__deepnote_big_number__()


In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Einspeisung in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{grid_feed_in_sum}"
    })

__deepnote_big_number__()


In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Lastspitze in kW {{peak_date}}')

    return json.dumps({
        "title": rendered_title,
        "value": f"{peak}"
    })

__deepnote_big_number__()


# 

### Erzeugungskennzahlen

In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Gesamterzeugung in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{total_production_sum}"
    })

__deepnote_big_number__()


In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('PV Gesamterzeugung in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{pv_production_sum}"
    })

__deepnote_big_number__()


In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('KWK Gesamterzeugung in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{chp_production_sum}"
    })

__deepnote_big_number__()


In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Wind Gesamterzeugung in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{wind_production_sum}"
    })

__deepnote_big_number__()


### Verbrauchskennzahlen

In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Brutto Stromverbrauch kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{mid_consumption_sum}"
    })

__deepnote_big_number__()


In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Eigenverbrauch in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{prod_self_consumption_sum}"
    })

__deepnote_big_number__()


# 

# 

### Bilanzkennzahlen

In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Autarkiegrad (%)')

    return json.dumps({
        "title": rendered_title,
        "value": f"{prod_self_consumption_share}"
    })

__deepnote_big_number__()


In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Eigenverbrauchquote (%)')

    return json.dumps({
        "title": rendered_title,
        "value": f"{prod_self_production_share}"
    })

__deepnote_big_number__()


# 

# 

### Batterie

In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Production to Battery Flow')

    return json.dumps({
        "title": rendered_title,
        "value": f"{production_to_battery_sum}"
    })

__deepnote_big_number__()


In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Grid to Battery Flow')

    return json.dumps({
        "title": rendered_title,
        "value": f"{grid_to_battery_sum}"
    })

__deepnote_big_number__()


In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Battery to Grid Flow')

    return json.dumps({
        "title": rendered_title,
        "value": f"{battery_to_grid_sum}"
    })

__deepnote_big_number__()


In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Battery to Consumption Flow')

    return json.dumps({
        "title": rendered_title,
        "value": f"{battery_to_consumption_sum}"
    })

__deepnote_big_number__()


Lastgang im zeitlichen Verlauf: 
Hier werden die PV\-Gesamtleistung, der Verbrauch \(Last\), der Netzbezug, die \(PV\) Einspeisung und die Batterie\-Gesamtleistung im ausgewählten Zeitraum dargestellt\.

## Energiebezug
Hier sehen Sie das Verhältnis von Eigenverbrauch und Netzbezug in dem von Ihnen gewählten Zeitraum an.

In [ ]:
battery_usecase_df = build_overview_df(energy_report_df, component_types=component_types)
cols_list = [col for col in battery_usecase_df.columns.tolist() if col not in ["grid_feed_in"]]

fig = plot_time_series_battery_soc_and_usecase(
    battery_usecase_df,
    cols=cols_list, 
    time_col="timestamp",
    battery_power_flow="battery_power_flow",
    soc_pct="battery_soc_pct",
    secondary_y_cols=["day_ahead_price"],
    secondary_y_title="EUR/MWh",
    title="Lastgang Übersicht",
    dotted_cols=[
        "grid_consumption_without_battery",
        "peak_before_optimization",
        "day_ahead_price",
    ],
    stack_mode="psc",
    xaxis_title="Zeitpunkt",
    yaxis_title="kW",
    soc_secondary_y_title="SOC [%]",
)

fig.show()

In [ ]:
energy_summary_df = compute_energy_summary(energy_report_df, resolution=processed_resolution)
energy_summary_df["Energy Source"] = energy_summary_df["Energy Source"].replace({'CHP': 'BHKW', 'Grid Import': 'Netzbezug'})
energy_summary_df = energy_summary_df.rename(columns={"Energy Source": "Energiebezug", "Energy [kWh]": "Energie [kWh]"})
energy_summary_df

In [ ]:
energy_pie_plot = pl.plot_energy_pie_chart(energy_summary_df, color_dict=color_dict)
energy_pie_plot.show()

## PV

PV\-Leistung 
Hier sehen Sie sowohl die kumulierte Leistung der PV\-Anlagen als auch die gestapelten Einzelleistungen der jeweiligen PV\-Anlagen\. Die Netzanschluss\-Kurve zeigt an, wie viel Stromüberschüsse ins Netz eingespeist wurden \(Kurve im negativen Bereich\)\. Beziehungsweise wieviel Strom vom Netz bezogen wurde \(Kurve im positiven Bereich\)\.
Um einzelne PV\-Anlagen zu filtern muss zunächst "Alle" aus dem Filter entfernt und dann die gewünschten PV IDs ausgewählt werden\.

In [ ]:
pv_filter_options = []
pv_columns = [c for c in energy_report_df.columns if "PV #" in c]
if 'pv' in component_types:
    pv_grid_filter_options = ["PV und Netzanschluss", "Nur PV", "Nur Netzanschluss"]
    pv_filter_options = ['Alle'] + [pv[3:] for pv in pv_columns]

In [ ]:
available_pv_units = ['Alle']

In [ ]:
pv_component_type = 'meter'

In [ ]:
Filter_PV = False

In [ ]:
if available_pv_units or Filter_PV:
    pv_analyse_df, pv_prod_filter_sum, pv_filter_text = assemble_component_analysis(
        component_filter=available_pv_units,
        component_key="pv",
        component_types=component_types,
        energy_report_df=energy_report_df,
        timestep_hours=timestep_hours,
        mapper=mapper,
        component_label="PV",
        value_col_name="pv_asset_production",
        invert_sign=True,
        trunc_values=True,
        mcfg=mcfg,
        component_id_source=pv_component_type,
    )
else:
    pv_analyse_df = pd.DataFrame()
pv_analyse_df

In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('PV Gesamterzeugung ({{pv_filter_text}}) in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{pv_prod_filter_sum}"
    })

__deepnote_big_number__()


In [ ]:
if available_pv_units or Filter_PV:
    if "pv" in component_types:
        fig = plot_time_series(
            pv_analyse_df,
            time_col="Zeitpunkt",
            title="PV Leistung [KWh]",
            yaxis_title="kWh",
            xaxis_title="Zeitpunkt",
            legend_title=None,
            long_format_flag=True,
            category_col="PV",
            value_col="PV-Erzeugung",
            color_dict=color_dict,
        )
        fig.show()

### Abweichungen zwischen Zähler\-/Wechselrichterwerten und Aggregaten

In [ ]:
if 'pv' in component_types:

    pv_component_ids = {
        str(i)
        for i in mcfg.component_type_ids("pv", component_category=pv_component_type) or []
    }

    pv_component_cols = [
        c for c in energy_report_df.columns
        if c.startswith("PV #") and component_id(c, "PV") in pv_component_ids
    ]

    debug_df = energy_report_df[
        ["timestamp", "pv_asset_production", *pv_component_cols]
    ].copy()

    debug_df["pv_component_sum"] = (
        -debug_df[pv_component_cols].clip(upper=0).sum(axis=1, min_count=1)
        if pv_component_cols
        else np.nan
    )

    debug_df[f"diff_vs_{pv_component_type}"] = (
        debug_df["pv_asset_production"] - debug_df["pv_component_sum"]
    )

    mismatch_df = debug_df[
        (
            ~np.isclose(
                debug_df[f"diff_vs_{pv_component_type}"].fillna(np.inf),
                0,
                atol=1e-3,
            )
        )
        & (debug_df["pv_asset_production"] != 0)
    ].copy()

else:
    mismatch_df = None
mismatch_df

## Batterie

Batterieleistungskurve 
Die Batterieleistungs\-Kurve zeigt kumuliert für alle Batterien an, wie viel Strom aus der Batterie bezogen wurde \(Kurve im negativen Bereich\)\. Beziehungsweise wieviel Strom in die Batterie eingespeist wurde \(Kurve im positiven Bereich\)\. 
Um einzelne Batterien zu filtern muss zunächst "Alle" aus dem Filter entfernt und dann die gewünschten Batterie IDs ausgewählt werden\.

In [ ]:
battery_filter_options = []
if 'battery' in component_types:
    battery_filter_options = ['Alle'] + [f'#{i}' for i in mcfg.component_type_ids('battery')]

In [ ]:
available_battery_units = ['Alle']

In [ ]:
battery_component_type = 'meter'

In [ ]:
Filter_Battery = False

In [ ]:
if available_battery_units or Filter_Battery:
    bat_analyse_df, bat_prod_filter_sum, bat_filter_text = assemble_component_analysis(
        component_filter=available_battery_units,
        component_key="battery",
        component_types=component_types,
        energy_report_df=energy_report_df,
        timestep_hours=timestep_hours,
        mapper=mapper,
        component_label="Battery",
        value_col_name="Batterie Leistungsfluss",
        mcfg=mcfg,
        component_id_source=battery_component_type,
    )

else:
    bat_analyse_df = pd.DataFrame()    

bat_analyse_df

In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Battery Energie ({{bat_filter_text}}) in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{bat_prod_filter_sum}"
    })

__deepnote_big_number__()


In [ ]:
if available_battery_units or Filter_Battery:
    if "battery" in {str(x).lower() for x in component_types}:
        fig = plot_time_series(
            bat_analyse_df,
            time_col="Zeitpunkt",
            title="Batterie Leistungsfluss [kWh]",
            yaxis_title="kWh",
            xaxis_title="Zeitpunkt",
            legend_title=None,
            long_format_flag=True,
            category_col="Battery",
            value_col="Batterie Leistungsfluss",
            color_dict=color_dict,
        )
        fig.show()

### Abweichungen zwischen Zähler\-/Wechselrichterwerten und Aggregaten

In [ ]:
if 'battery' in component_types:
    battery_component_ids = {
        str(i)
        for i in mcfg.component_type_ids("battery", component_category=battery_component_type) or []
    }

    battery_component_cols = [
        c for c in energy_report_df.columns
        if c.startswith("Battery #") and component_id(c, "Battery") in battery_component_ids
    ]

    debug_df = energy_report_df[
        ["timestamp", "battery_power_flow", *battery_component_cols]
    ].copy()

    debug_df["battery_component_sum"] = (
        debug_df[battery_component_cols].sum(axis=1, min_count=1)
        if battery_component_cols
        else np.nan
    )

    debug_df[f"diff_vs_{battery_component_type}"] = (
        debug_df["battery_power_flow"] - debug_df["battery_component_sum"]
    )

    mismatch_df = debug_df[
        (
            ~np.isclose(
                debug_df[f"diff_vs_{battery_component_type}"].fillna(np.inf),
                0,
                atol=1e-3,
            )
        )
        & (debug_df["battery_power_flow"] != 0)
    ].copy()

else: 
    mismatch_df = None
mismatch_df

# KWK

In [ ]:
chp_filter_options = []
if 'chp' in component_types:
    chp_filter_options = ['Alle'] + [f'#{i}' for i in mcfg.component_type_ids('chp')]

In [ ]:
available_chp_units = ['Alle']

In [ ]:
kwk_component_type = 'meter'

In [ ]:
Filter_KWK = False

In [ ]:
if available_chp_units or Filter_BHKW:
    chp_analyse_df, chp_prod_filter_sum, chp_filter_text = assemble_component_analysis(
        component_filter=available_chp_units,
        component_key="chp",
        component_types=component_types,
        energy_report_df=energy_report_df,
        timestep_hours=timestep_hours,
        mapper=mapper,
        component_label="CHP",
        value_col_name="chp_asset_production",
        invert_sign=True,
        trunc_values=True,
        mcfg=mcfg,
        component_id_source=kwk_component_type,
    )
    chp_analyse_df = chp_analyse_df.rename(columns={"CHP": "KWK"})

else:
    chp_analyse_df = pd.DataFrame()

chp_analyse_df

In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('KWK Gesamterzeugung ({{chp_filter_text}}) in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{chp_prod_filter_sum}"
    })

__deepnote_big_number__()


In [ ]:
if available_chp_units or Filter_BHKW:
    if "chp" in {str(x).lower() for x in component_types}:
        fig = plot_time_series(
            chp_analyse_df,
            time_col="Zeitpunkt",
            title="KWK Erzeugung [kWh]",
            yaxis_title="kWh",
            xaxis_title="Zeitpunkt",
            legend_title=None,
            long_format_flag=True,
            category_col="KWK",
            value_col="KWK-Erzeugung",
            color_dict=color_dict
        )
        fig.show()

### Abweichungen zwischen Zähler\-/Wechselrichterwerten und Aggregaten

In [ ]:
if 'chp' in component_types:
    chp_component_ids = {
        str(i)
        for i in mcfg.component_type_ids("chp", component_category=kwk_component_type) or []
    }

    chp_component_cols = [
        c for c in energy_report_df.columns
        if c.startswith("CHP #") and component_id(c) in chp_component_ids
    ]

    debug_df = energy_report_df[
        ["timestamp", "kwk_asset_production", *chp_component_cols]
    ].copy()

    debug_df["kwk_component_sum"] = (
        -debug_df[chp_component_cols].clip(upper=0).sum(axis=1, min_count=1)
        if chp_component_cols
        else np.nan
    )

    debug_df[f"diff_vs_{kwk_component_type}"] = (
        debug_df["kwk_asset_production"] - debug_df["kwk_component_sum"]
    )

    mismatch_df = debug_df[
        (
            ~np.isclose(
                debug_df[f"diff_vs_{kwk_component_type}"].fillna(np.inf),
                0,
                atol=1e-3,
            )
        )
        & (debug_df["kwk_asset_production"] != 0)
    ].copy()

else:
    mismatch_df = None    

mismatch_df

# Wind

In [ ]:
wind_filter_options = []
if 'wind' in component_types:
    wind_filter_options = ['Alle'] + [f'#{i}' for i in mcfg.component_type_ids('wind')]

In [ ]:
available_wind_units = ['Alle']

In [ ]:
wind_component_type = 'meter'

In [ ]:
Filter_Wind = False

In [ ]:
if available_wind_units or Filter_Wind:
    wind_analyse_df, wind_prod_filter_sum, wind_filter_text = assemble_component_analysis(
        component_filter=available_wind_units,
        component_key="wind",
        component_types=component_types,
        energy_report_df=energy_report_df,
        timestep_hours=timestep_hours,
        mapper=mapper,
        component_label="Wind",
        value_col_name="wind_asset_production",
        invert_sign=True,
        trunc_values=True,
        mcfg=mcfg,
        component_id_source=wind_component_type,
    )
else:
    wind_analyse_df = pd.DataFrame()

wind_analyse_df

In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('Wind Gesamterzeugung ({{wind_filter_text}}) in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{wind_prod_filter_sum}"
    })

__deepnote_big_number__()


In [ ]:
if available_wind_units or Filter_Wind:
    if "wind" in {str(x).lower() for x in component_types}:
        fig = plot_time_series(
            wind_analyse_df,
            time_col="Zeitpunkt",
            title="Wind Erzeugung [kWh]",
            yaxis_title="kWh",
            xaxis_title="Zeitpunkt",
            legend_title=None,
            long_format_flag=True,
            category_col="Wind",
            value_col="Wind-Erzeugung",
            color_dict=color_dict
        )
        fig.show()

### Abweichungen zwischen Zähler\-/Wechselrichterwerten und Aggregaten

In [ ]:
if 'wind' in component_types:
    wind_component_ids = {
        str(i)
        for i in mcfg.component_type_ids("wind", component_category=wind_component_type) or []
    }

    wind_component_cols = [
        c for c in energy_report_df.columns
        if c.startswith("Wind #") and component_id(c) in wind_component_ids
    ]

    debug_df = energy_report_df[
        ["timestamp", "wind_asset_production", *wind_component_cols]
    ].copy()

    debug_df["wind_component_sum"] = (
        -debug_df[wind_component_cols].clip(upper=0).sum(axis=1, min_count=1)
        if wind_component_cols
        else np.nan
    )

    debug_df[f"diff_vs_{wind_component_type}"] = (
        debug_df["wind_asset_production"] - debug_df["wind_component_sum"]
    )

    mismatch_df = debug_df[
        (
            ~np.isclose(
                debug_df[f"diff_vs_{wind_component_type}"].fillna(np.inf),
                0,
                atol=1e-3,
            )
        )
        & (debug_df["wind_asset_production"] != 0)
    ].copy()

else:
    mismatch_df = None
mismatch_df

# EV Charger

In [ ]:
ev_filter_options = []
if 'ev' in component_types:
    ev_filter_options = ['Alle'] + [f'#{i}' for i in mcfg.component_type_ids('ev')]

In [ ]:
available_ev_units = []

In [ ]:
Filter_EV = False

In [ ]:
if available_ev_units or Filter_EV:
    ev_analyse_df, ev_prod_filter_sum, ev_filter_text = assemble_component_analysis(
        component_filter=available_ev_units,
        component_key="ev",
        component_types=component_types,
        energy_report_df=energy_report_df,
        timestep_hours=timestep_hours,
        mapper=mapper,
        component_label="EV",
        value_col_name="ev_asset_production",
        invert_sign=True,
        trunc_values=True,
        mcfg=mcfg,
        component_id_source=battery_component_type,
    )

ev_analyse_df

In [ ]:

def __deepnote_big_number__():
    import json
    import jinja2
    from jinja2 import meta

    def render_template(template):
        parsed_content = jinja2.Environment().parse(template)

        required_variables = meta.find_undeclared_variables(parsed_content)

        context = {
            variable_name: globals().get(variable_name)
            for variable_name in required_variables
        }

        result = jinja2.Environment().from_string(template).render(context)

        return result

    rendered_title = render_template('EV Gesamterzeugung ({{ev_filter_text}}) in kWh')

    return json.dumps({
        "title": rendered_title,
        "value": f"{ev_prod_filter_sum}"
    })

__deepnote_big_number__()


In [ ]:
if available_ev_units or Filter_EV:
    if "ev" in {str(x).lower() for x in component_types}:
        fig = plot_time_series(
            ev_analyse_df,
            time_col="Zeitpunkt",
            title="EV Erzeugung [kWh]",
            yaxis_title="kWh",
            xaxis_title="Zeitpunkt",
            legend_title=None,
            long_format_flag=True,
            category_col="EV",
            value_col="EV-Erzeugung",
            color_dict=color_dict
        )
        fig.show()

## Daten

In [ ]:
print(f'{datetime.strftime(start_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")} - {datetime.strftime(end_date.astimezone(ZoneInfo("CET")).date(), "%d.%m.%Y")}')

### Daten exportieren

Die Daten werden unter angegebenem Dateinamen in den Deepnote Files gespeichert und können von dort heruntergeladen werden\.
Mehr erfahren

In [ ]:
output = 'Lamerdingen_2026_07.csv'

In [ ]:
download_data = False

In [ ]:
if output != "" and energy_report_df is not None and download_data:
    output_df = mapper.to_display(energy_report_df)
    output_df = output_df.round(3)
    if os.path.exists(output):
        answer = input(f"Die Datei '{output}' existiert bereits. Überschreiben? (j/n): ").strip().lower()

        if answer not in ["j", "ja", "y", "yes"]:
            print("Export abgebrochen - Datei wurde nicht überschrieben.")
        else:
            if output.endswith(".csv"):
                output_df.to_csv(output, index=False)
                print(f"Die Daten wurden erfolgreich in {output} exportiert.")
            elif output.endswith(".xlsx"):
                output_df.to_excel(output, index=False)
                print(f"Die Daten wurden erfolgreich in {output} exportiert.")
            else:
                print("Fehler: Output muss .csv oder .xlsx sein.")

    elif output.endswith(".csv"):
        output_df.to_csv(output, index=False)
        print(f"Die Daten wurden erfolgreich in {output} exportiert.")

    elif output.endswith(".xlsx"):
        output_df.to_excel(output, index=False)
        print(f"Die Daten wurden erfolgreich in {output} exportiert.")

    else:
        print("Fehler: Output muss .csv oder .xlsx sein.")

else:
    output_df = pd.DataFrame()

output_df

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4c1f1e91-3e70-466c-b7cc-b35cf9e17cac' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>